<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session08_modern_cnn_architectures_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۸: معماری‌های مدرن CNN: از LeNet تا ResNet

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session08_modern_cnn_architectures.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1, residual=True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)
        return F.relu(out)


class CifarResNet(nn.Module):
    def __init__(self, n=3, residual=True, n_classes=10, block=BasicBlock):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for stage, c_out in enumerate([16, 32, 64]):
            for i in range(n):
                stride = 2 if (stage > 0 and i == 0) else 1
                layers.append(block(c_in, c_out, stride, residual))
                c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, n_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        return self.head(self.layers(self.stem(x)))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

تعداد پارامترهای یک بلوک VGG با سه لایه ۳×۳ و ۲۵۶ کانال را با یک لایه ۷×۷ با همان کانال‌ها مقایسه کنید.

</div>

In [ ]:
C = 256
three_3x3 = nn.Sequential(*[nn.Conv2d(C, C, 3, padding=1) for _ in range(3)])
one_7x7 = nn.Conv2d(C, C, 7, padding=3)
print(f'three 3x3 layers: 3 * 256 * (256*9 + 1)  = {3 * C * (C * 9 + 1):,} | code: {count_params(three_3x3):,}')
print(f'one 7x7 layer   :     256 * (256*49 + 1) = {C * (C * 49 + 1):,} | code: {count_params(one_7x7):,}')
print(f'ratio: {count_params(one_7x7) / count_params(three_3x3):.2f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

سه لایهٔ ۳×۳ پشت سر هم همان **میدان دید ۷×۷** را دارند، اما با $27C^2$ پارامتر به‌جای $49C^2$ (حدود ۴۵٪ کمتر). علاوه بر این، بین آن‌ها سه تابع فعال‌سازی غیرخطی قرار می‌گیرد، پس تابعی که می‌توانند بسازند غنی‌تر است. این ایدهٔ اصلی VGG بود و از آن به بعد کرنل ۳×۳ انتخاب استاندارد شد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

یک **Bottleneck Block** (۱×۱ ← ۳×۳ ← ۱×۱ با ضریب گسترش ۴) پیاده‌سازی کنید و تعداد پارامتر آن را با دو BasicBlock هم‌خروجی مقایسه کنید.

</div>

In [ ]:
class Bottleneck(nn.Module):
    """1x1 (reduce) -> 3x3 -> 1x1 (expand by 4), as in ResNet-50/101/152."""
    expansion = 4

    def __init__(self, c_in, width, stride=1):
        super().__init__()
        c_out = width * self.expansion
        self.body = nn.Sequential(
            nn.Conv2d(c_in, width, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU(),
            nn.Conv2d(width, width, 3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(width), nn.ReLU(),
            nn.Conv2d(width, c_out, 1, bias=False), nn.BatchNorm2d(c_out),
        )
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        return F.relu(self.body(x) + self.shortcut(x))


x = torch.randn(2, 256, 14, 14)
bottleneck = Bottleneck(256, 64)                                  # 256 -> 64 -> 64 -> 256
two_basic = nn.Sequential(BasicBlock(256, 256), BasicBlock(256, 256))
print('output shapes:', tuple(bottleneck(x).shape), tuple(two_basic(x).shape))
print(f'one Bottleneck block : {count_params(bottleneck):,} parameters (3 conv layers)')
print(f'two BasicBlocks      : {count_params(two_basic):,} parameters (4 conv layers)')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بلوک Bottleneck تعداد کانال‌ها را با کانولوشن ۱×۱ از ۲۵۶ به ۶۴ کم می‌کند، کانولوشن پرهزینهٔ ۳×۳ را روی همین ۶۴ کانال انجام می‌دهد و دوباره با ۱×۱ به ۲۵۶ برمی‌گرداند. نتیجه: با حدود ۷۰ هزار پارامتر، به‌جای بیش از ۱٫۱ میلیون در دو BasicBlock، خروجی هم‌اندازه می‌سازد. همین صرفه‌جویی (حدود ۱۶ برابر) امکان ساخت ResNet-50 و ResNet-152 را فراهم کرد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

آزمایش بخش ۶ را با $n = 9$ (شبکه‌های ۵۶ لایه) تکرار کنید. فاصله Plain و ResNet چه تغییری می‌کند؟

</div>

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
train_loader = DataLoader(datasets.CIFAR10('data', train=True, download=True, transform=train_tf),
                          batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(datasets.CIFAR10('data', train=False, download=True, transform=test_tf),
                         batch_size=512, num_workers=2, pin_memory=True)
CIFAR_CLASSES = test_loader.dataset.classes

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هر چهار شبکه (Plain و ResNet با عمق ۲۰ و ۵۶) را با تنظیمات یکسان آموزش می‌دهیم. روی T4 هر مدل ۵۶ لایه چند ده دقیقه طول می‌کشد.

</div>

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None):
    """One pass over the training data. `scheduler` is stepped after every batch if given."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, optimizer, epochs, device, scheduler=None, verbose=True):
    """Train for `epochs` epochs and return the history of losses and accuracies."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
        va = evaluate(model, val_loader, criterion, device)
        for k, v in zip(history, (*tr, *va)):
            history[k].append(v)
        if verbose:
            print(f'epoch {epoch:2d} | train loss {tr[0]:.4f} acc {tr[1]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f}')
    return history

In [ ]:
EPOCHS = 30


def train_cifar(model, epochs=EPOCHS):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, epochs=epochs, steps_per_epoch=len(train_loader))
    start = time.time()
    history = fit(model, train_loader, test_loader, optimizer, epochs, device, scheduler=scheduler, verbose=False)
    print(f'  {time.time() - start:.0f}s | final train acc {history["train_acc"][-1]:.4f} | test acc {history["val_acc"][-1]:.4f}')
    return history


results, models = {}, {}
for n in (3, 9):
    for residual in (False, True):
        name = f'{"ResNet" if residual else "Plain"}-{6 * n + 2}'
        print(name)
        torch.manual_seed(0)
        models[name] = CifarResNet(n=n, residual=residual)
        results[name] = train_cifar(models[name])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in results.items():
    style = '-' if 'ResNet' in name else '--'
    axes[0].plot(h['train_loss'], style, label=name); axes[1].plot(h['val_acc'], style, label=name)
axes[0].set_title('training loss'); axes[1].set_title('test accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend()
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با عمق ۲۰، فاصلهٔ Plain و ResNet کوچک است. با عمق ۵۶ این فاصله **بیشتر** می‌شود: Plain-56 حتی **Loss آموزش** بالاتری از Plain-20 دارد. یعنی مشکل Overfitting نیست، بلکه بهینه‌سازی شبکهٔ ساده و عمیق سخت است (مسئلهٔ **Degradation** در مقالهٔ ResNet). در مقابل، ResNet-56 دست‌کم به خوبی ResNet-20 آموزش می‌بیند و معمولاً دقت بالاتری دارد؛ چون هر بلوک اضافه در بدترین حالت می‌تواند تقریباً همانی شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

در `BasicBlock` جای ReLU آخر را عوض کنید و آن را قبل از جمع با میان‌بر قرار دهید. این تغییر چه اثری بر خاصیت «مسیر مستقیم گرادیان» دارد؟ (راهنمایی: مقاله Pre-activation ResNet از He et al., 2016.)

</div>

In [ ]:
class ReLUBeforeAdd(BasicBlock):
    """Variant: ReLU applied to the residual branch *before* the addition; nothing after it."""

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        return out + self.shortcut(x)


def first_layer_grad_norm(model):
    model.train()
    torch.manual_seed(1)
    x, y = torch.randn(32, 3, 32, 32), torch.randint(0, 10, (32,))
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    return model.stem[0].weight.grad.norm().item()


for n in (3, 9, 18):
    torch.manual_seed(0)
    original = CifarResNet(n, block=BasicBlock)
    torch.manual_seed(0)
    variant = CifarResNet(n, block=ReLUBeforeAdd)
    print(f'depth {6 * n + 2:3d}: grad norm original = {first_layer_grad_norm(original):.2e} | '
          f'ReLU before add = {first_layer_grad_norm(variant):.2e}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در بلوک اصلی خروجی $\text{ReLU}(F(x) + x)$ است؛ پس مسیر میان‌بر دقیقاً همانی نیست و از یک ReLU عبور می‌کند. بعد از $L$ بلوک، گرادیان از $L$ تابع ReLU می‌گذرد که بخشی از آن را صفر می‌کنند. در نسخهٔ جدید خروجی $x + \text{ReLU}(F(x))$ است و مسیر $x_{l+1} = x_l + \ldots$ از ورودی تا خروجی **کاملاً خطی و بدون مانع** است:

$$x_L = x_l + \sum_{i=l}^{L-1} \mathcal{F}_i(x_i) \quad\Rightarrow\quad \frac{\partial x_L}{\partial x_l} = I + \frac{\partial}{\partial x_l}\sum_i \mathcal{F}_i$$

یعنی گرادیان همیشه یک جملهٔ همانی دارد که مستقیم به لایه‌های اول می‌رسد. اما این نسخه اشکال دارد: شاخهٔ باقی‌مانده بعد از ReLU همیشه **نامنفی** است. پس هر بلوک فقط می‌تواند به ویژگی‌ها چیزی **اضافه** کند و مقدارشان در طول شبکه مدام بزرگ‌تر می‌شود. He et al. (2016) راه‌حل بهتر را در **Pre-activation ResNet** پیشنهاد کردند: ترتیب BN → ReLU → Conv را **قبل** از کانولوشن‌ها بگذاریم و بعد از جمع هیچ عملی نباشد. این کار هم مسیر همانی تمیز را حفظ می‌کند و هم شاخهٔ باقی‌مانده می‌تواند علامت دلخواه داشته باشد. با این تغییر، آموزش ResNet با بیش از ۱۰۰۰ لایه ممکن شد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

Confusion Matrix مدل ResNet-20 را رسم کنید و آن را با مدل `CifarCNN` جلسه ۷ مقایسه کنید.

</div>

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))


class CifarCNN(nn.Module):                     # the model of session 7
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 64), conv_block(64, 128), conv_block(128, 256))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(0)
cifar_cnn = CifarCNN().to(device)
opt = torch.optim.AdamW(cifar_cnn.parameters(), lr=1e-3, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=15, steps_per_epoch=len(train_loader))
fit(cifar_cnn, train_loader, test_loader, opt, 15, device, scheduler=sched, verbose=False)


@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, labels = [], []
    for X, y in loader:
        preds.append(model(X.to(device)).argmax(1).cpu()); labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, (name, m) in zip(axes, [('ResNet-20', models['ResNet-20']), ('CifarCNN (session 7)', cifar_cnn)]):
    p, l = predict_all(m, test_loader)
    ConfusionMatrixDisplay(confusion_matrix(l, p), display_labels=CIFAR_CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f'{name}: accuracy {np.mean(p == l):.4f}, parameters {count_params(m):,}')
plt.tight_layout(); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ResNet-20 با حدود ۲۷۰ هزار پارامتر، یعنی تقریباً یک‌چهارم پارامترهای `CifarCNN` (حدود ۱٫۱ میلیون)، به دقتی بالاتر یا هم‌سطح می‌رسد. الگوی خطاها در هر دو مدل شبیه است: گربه و سگ بیشترین اشتباه را دارند و پس از آن حیوانات چهارپا و خودرو و کامیون. این خطاها ذاتی داده و وضوح ۳۲×۳۲ است؛ معماری بهتر تعدادشان را کم می‌کند، اما جایشان را عوض نمی‌کند.

</div>